# AgroPredix - Limpieza, EDA e integracion de etiquetas
Fuente: SISAGRI / MIDAGRI, Arequipa (registro mensual por distrito y cultivo)

## 0. Configuracion

In [ ]:
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

URL = "https://raw.githubusercontent.com/AldoVasquez07/st2-rec-doc-2026/refs/heads/main/Dataset/sisagri_arequipa.csv"

# Parametros de calidad y etiquetado
MES_INICIO_CAMPANA = 8      # campana agricola: agosto - julio
MIN_HA = 3                  # area minima (ha) para considerar confiable un rendimiento (los datos vienen redondeados a ha enteras)
MIN_REG_PRODUCTO = 15       # registros minimos por cultivo para fijar umbrales de etiqueta

## 1. Carga de datos

In [ ]:
raw = pd.read_csv(URL, encoding="utf-8-sig")
print(raw.shape)
raw.head()

## 2. Inspeccion inicial

In [ ]:
print(raw.dtypes, "\n")
print("Nulos por columna:\n", raw.isna().sum(), "\n")
raw.describe().T

## 3. Limpieza

### 3.1 Nombres de columnas y tipos

In [ ]:
df = raw.copy()
df.columns = (df.columns.str.strip().str.lower()
              .str.replace("\u00f1", "n", regex=False))
df = df.rename(columns={"siembra": "siembra_ha", "cosecha": "cosecha_ha", "produccion": "produccion_t"})

for c in ["departamento", "provincia", "distrito", "producto"]:
    df[c] = df[c].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)

df["cod_ubigeo"] = df["cod_ubigeo"].astype(str).str.zfill(6)       # ubigeo de 6 digitos
df["cod_producto"] = df["cod_producto"].astype(str)
df.dtypes

### 3.2 Duplicados y consistencia de claves

In [ ]:
clave = ["ano", "mes", "cod_ubigeo", "cod_producto"]
print("Filas duplicadas exactas:", df.duplicated().sum())
print("Claves duplicadas:", df.duplicated(clave).sum())
print("Ubigeos con >1 distrito:", (df.groupby("cod_ubigeo").distrito.nunique() > 1).sum())
print("Codigos de producto con >1 nombre:", (df.groupby("cod_producto").producto.nunique() > 1).sum())
df = df.drop_duplicates(clave)

### 3.3 Normalizacion de nombres de producto

In [ ]:
mapa = df[["cod_producto", "producto"]].drop_duplicates().copy()
mapa["base"] = (mapa.producto.str.replace(r"\s*\(.*?\)", "", regex=True)
                .str.replace("MANDARIN0", "MANDARINO").str.strip())
# si al quitar parentesis dos codigos distintos colisionan, se conserva el nombre completo
colision = mapa.groupby("base").cod_producto.transform("nunique") > 1
mapa["producto_limpio"] = np.where(colision, mapa.producto, mapa.base)

df = df.merge(mapa[["cod_producto", "producto_limpio"]], on="cod_producto", how="left")
df = df.drop(columns="producto").rename(columns={"producto_limpio": "producto"})
print("Productos:", df.producto.nunique())
mapa[mapa.producto != mapa.producto_limpio][["producto", "producto_limpio"]]

### 3.4 Fecha y campana agricola

In [ ]:
df["fecha"] = pd.to_datetime(dict(year=df.ano, month=df.mes, day=1))
df["campana_ini"] = np.where(df.mes >= MES_INICIO_CAMPANA, df.ano, df.ano - 1)
df["campana"] = df.campana_ini.astype(str) + "-" + (df.campana_ini + 1).astype(str)

# una campana es completa si tiene sus 12 meses en el dataset
meses_camp = df.groupby("campana").fecha.nunique().rename("n_meses")
meses_camp = meses_camp.to_frame().assign(completa=lambda d: d.n_meses == 12)
print("Rango:", df.fecha.min().date(), "a", df.fecha.max().date())
meses_camp

### 3.5 Valores invalidos y variables derivadas

In [ ]:
print("verde_actual negativo:", (df.verde_actual < 0).sum())
print("produccion > 0 con cosecha_ha = 0:", ((df.produccion_t > 0) & (df.cosecha_ha == 0)).sum())
print("tasa_perdida < 0:", (df.tasa_perdida < 0).sum())

df.loc[df.verde_actual < 0, "verde_actual"] = np.nan

# rendimiento: se recalcula; si hay produccion pero la cosecha fue redondeada a 0 ha, queda indefinido
df["rendimiento_t_ha"] = np.where(df.cosecha_ha > 0, df.produccion_t / df.cosecha_ha.replace(0, np.nan), np.nan)

# tasa_perdida original es un flujo mensual (siembra y cosecha ocurren en meses distintos): se descarta
df = df.drop(columns="tasa_perdida")

# precio en S/ por kg; ingreso en soles
df["ingreso_soles"] = df.precio_chacra * df.produccion_t * 1000

### 3.6 Etiquetas de cultivo (grupo, tipo y familia botanica)

In [ ]:
GRUPOS = {
    "1201": ("pasto_cultivado", "pasto"),
    "1301": ("frutal", "permanente"), "1302": ("frutal", "permanente"),
    "1401": ("cereal_grano_andino", "transitorio"),
    "1402": ("melon_sandia", "transitorio"),
    "1403": ("hortaliza", "transitorio"),
    "1404": ("raiz_tuberculo", "transitorio"),
    "1405": ("forraje_anual", "transitorio"),
    "1406": ("leguminosa_grano_seco", "transitorio"),
    "1407": ("otro_transitorio", "transitorio"),
    "1501": ("perenne_semiperenne", "permanente"),
    "1502": ("industrial_especial", "permanente"),
}
FAMILIA = {
    # Poaceae
    "ARROZ": "Poaceae", "CEBADA GRANO": "Poaceae", "CEBADA FORRAJERA": "Poaceae", "MAIZ AMARILLO DURO": "Poaceae",
    "MAIZ AMILACEO": "Poaceae", "MAIZ CHOCLO": "Poaceae", "MAIZ CHALA": "Poaceae", "MAIZ MORADO": "Poaceae",
    "SORGO GRANO": "Poaceae", "TRIGO BLANDO": "Poaceae", "AVENA FORRAJERA": "Poaceae",
    "DACTILYS": "Poaceae", "RYE GRASS": "Poaceae",
    # Amaranthaceae
    "ACHITA, KIWICHA, O AMARANTO": "Amaranthaceae", "CA\u00d1AHUA O CANIHUA": "Amaranthaceae", "QUINUA": "Amaranthaceae",
    "ACELGA": "Amaranthaceae", "BETARRAGA": "Amaranthaceae",
    # Fabaceae
    "ARVEJA GRANO VERDE": "Fabaceae", "ARVEJA GRANO SECO": "Fabaceae", "FRIJOL VAINITA": "Fabaceae",
    "FRIJOL GRANO SECO": "Fabaceae", "HABA GRANO VERDE": "Fabaceae", "HABA GRANO SECO": "Fabaceae",
    "PALLAR GRANO SECO": "Fabaceae", "ALFALFA": "Fabaceae", "TREBOL": "Fabaceae",
    # Solanaceae
    "PAPA": "Solanaceae", "TOMATE": "Solanaceae", "AJI": "Solanaceae", "PAPRIKA": "Solanaceae",
    # Amaryllidaceae
    "AJO": "Amaryllidaceae", "CEBOLLA": "Amaryllidaceae", "CEBOLLA CABEZA AMARILLA": "Amaryllidaceae",
    "CEBOLLA CHINA": "Amaryllidaceae", "PORO": "Amaryllidaceae",
    # Cucurbitaceae
    "MELON": "Cucurbitaceae", "SANDIA": "Cucurbitaceae", "ZAPALLO": "Cucurbitaceae", "CALABAZA": "Cucurbitaceae",
    "CAIGUA": "Cucurbitaceae", "PEPINILLO": "Cucurbitaceae",
    # Brassicaceae
    "BROCOLI": "Brassicaceae", "COL O REPOLLO": "Brassicaceae", "COLIFLOR": "Brassicaceae",
    "NABO": "Brassicaceae", "RABANO": "Brassicaceae",
    # Apiaceae
    "APIO": "Apiaceae", "ZANAHORIA": "Apiaceae",
    # Otras
    "LECHUGA": "Asteraceae", "CAMOTE": "Convolvulaceae", "YUCA": "Euphorbiaceae", "OCA": "Oxalidaceae",
    "OLLUCO": "Basellaceae", "MASHUA O IZANO": "Tropaeolaceae", "CHIA": "Lamiaceae",
}
pref = df.cod_producto.str[:4]
df["grupo"] = pref.map(lambda p: GRUPOS.get(p, ("otro", "otro"))[0])
df["tipo_cultivo"] = pref.map(lambda p: GRUPOS.get(p, ("otro", "otro"))[1])
df["familia"] = df.producto.map(FAMILIA)
df["es_leguminosa"] = df.familia.eq("Fabaceae")

sin_fam = df[(df.tipo_cultivo.isin(["transitorio", "pasto"])) & df.familia.isna()].producto.unique()
print("Transitorios/pastos sin familia:", sin_fam)
df[["producto", "grupo", "tipo_cultivo", "familia"]].drop_duplicates().sort_values(["tipo_cultivo", "grupo"]).head(15)

### 3.7 Resumen de la limpieza

In [ ]:
mensual = df.sort_values(["fecha", "cod_ubigeo", "producto"]).reset_index(drop=True)
print("Filas:", len(mensual), "| Distritos:", mensual.cod_ubigeo.nunique(), "| Productos:", mensual.producto.nunique())
print("Nulos restantes:\n", mensual.isna().sum()[lambda s: s > 0])
mensual.head()

## 4. EDA sobre el registro mensual

### 4.1 Cobertura temporal y espacial

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
cob = mensual.groupby("fecha").size()
axs[0].bar(cob.index, cob.values, width=25, color="#3b7ea1")
axs[0].set_title("Registros por mes")
axs[0].set_ylabel("filas")

prov = mensual.groupby("provincia").cod_ubigeo.nunique().sort_values()
axs[1].barh(prov.index, prov.values, color="#6a9e4f")
axs[1].set_title("Distritos con registros por provincia")
plt.tight_layout(); plt.show()

### 4.2 Estructura de ceros

In [ ]:
num = ["siembra_ha", "cosecha_ha", "produccion_t", "verde_actual", "precio_chacra", "rendimiento_t_ha"]
ceros = pd.DataFrame({"pct_cero": (mensual[num] == 0).mean() * 100, "pct_nulo": mensual[num].isna().mean() * 100}).round(1)
ceros.plot.barh(figsize=(8, 3.5), color=["#c0803a", "#7a7a7a"])
plt.title("Porcentaje de ceros y nulos por variable"); plt.xlabel("%"); plt.show()
ceros

### 4.3 Superficie por cultivo

In [ ]:
top = (mensual[mensual.tipo_cultivo == "transitorio"].groupby("producto").siembra_ha.sum()
       .sort_values(ascending=False).head(15))
plt.figure(figsize=(9, 5))
sns.barplot(x=top.values, y=top.index, color="#4c9a2a")
plt.title("Top 15 cultivos transitorios por superficie sembrada acumulada (ha)")
plt.xlabel("ha"); plt.ylabel(""); plt.show()

### 4.4 Estacionalidad de siembra

In [ ]:
top12 = top.index[:12]
est = (mensual[mensual.producto.isin(top12)]
       .pivot_table(index="producto", columns="mes", values="siembra_ha", aggfunc="sum", fill_value=0)
       .reindex(top12))
est_n = est.div(est.sum(axis=1), axis=0)
plt.figure(figsize=(11, 5))
sns.heatmap(est_n, cmap="YlGn", annot=False, cbar_kws={"label": "proporcion de siembra anual"})
plt.title("Distribucion mensual de la siembra por cultivo"); plt.xlabel("mes"); plt.ylabel(""); plt.show()

### 4.5 Rendimiento y precio por cultivo (registros mensuales con cosecha)

In [ ]:
v = mensual[(mensual.tipo_cultivo == "transitorio") & (mensual.cosecha_ha >= 1) & mensual.producto.isin(top12)]
fig, axs = plt.subplots(1, 2, figsize=(16, 5))
sns.boxplot(data=v, y="producto", x="rendimiento_t_ha", order=top12, ax=axs[0], color="#8fc17a", fliersize=2)
axs[0].set_title("Rendimiento (t/ha)"); axs[0].set_ylabel("")
sns.boxplot(data=v[v.precio_chacra > 0], y="producto", x="precio_chacra", order=top12, ax=axs[1], color="#e0b36a", fliersize=2)
axs[1].set_xscale("log"); axs[1].set_title("Precio en chacra (S/ por kg, escala log)"); axs[1].set_ylabel("")
plt.tight_layout(); plt.show()

### 4.6 Serie de precios

In [ ]:
sel = list(top12[:6])
pr = (mensual[mensual.producto.isin(sel) & (mensual.precio_chacra > 0)]
      .groupby(["fecha", "producto"]).apply(lambda g: np.average(g.precio_chacra, weights=g.produccion_t)).unstack())
pr.plot(figsize=(12, 4), marker="o", ms=3)
plt.title("Precio en chacra ponderado por produccion (S/ por kg)"); plt.ylabel("S/ por kg"); plt.xlabel("")
plt.legend(ncol=3, fontsize=8); plt.show()

### 4.7 Correlaciones

In [ ]:
c = np.log1p(mensual[num].fillna(0)).corr()
plt.figure(figsize=(6.5, 5))
sns.heatmap(c, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1)
plt.title("Correlacion (log1p)"); plt.show()

## 5. Agregacion a campana agricola (unidad de analisis: distrito - cultivo - campana)

In [ ]:
llave = ["cod_ubigeo", "producto", "campana"]

camp = (mensual.groupby(llave, as_index=False)
        .agg(campana_ini=("campana_ini", "first"),
             siembra_ha=("siembra_ha", "sum"), cosecha_ha=("cosecha_ha", "sum"),
             produccion_t=("produccion_t", "sum"), ingreso_soles=("ingreso_soles", "sum"),
             verde_max=("verde_actual", "max"), n_meses_activo=("verde_actual", lambda s: (s > 0).sum())))

# rendimiento y precio ponderados a nivel campana
camp["rendimiento_t_ha"] = np.where(camp.cosecha_ha > 0, camp.produccion_t / camp.cosecha_ha.replace(0, np.nan), np.nan)
camp["precio_kg"] = np.where(camp.produccion_t > 0, camp.ingreso_soles / (camp.produccion_t * 1000), np.nan)
camp["ingreso_ha_soles"] = np.where(camp.cosecha_ha > 0, camp.ingreso_soles / camp.cosecha_ha.replace(0, np.nan), np.nan)

# mes pico de siembra
pico = (mensual[mensual.siembra_ha > 0].sort_values("siembra_ha", ascending=False)
        .drop_duplicates(llave)[llave + ["mes"]].rename(columns={"mes": "mes_pico_siembra"}))
camp = camp.merge(pico, on=llave, how="left")

# dimensiones
dim_dist = mensual[["cod_ubigeo", "provincia", "distrito"]].drop_duplicates("cod_ubigeo")
dim_prod = mensual[["producto", "cod_producto", "grupo", "tipo_cultivo", "familia", "es_leguminosa"]].drop_duplicates("producto")
camp = (camp.merge(dim_dist, on="cod_ubigeo").merge(dim_prod, on="producto")
        .merge(meses_camp.reset_index()[["campana", "completa"]].rename(columns={"completa": "campana_completa"}), on="campana"))
print(camp.shape)
camp.head()

## 6. Integracion de etiquetas

### 6.1 Registros validos

In [ ]:
# El area minima evita rendimientos inflados por redondeo; solo cultivos transitorios en campanas completas
camp["registro_valido"] = (camp.tipo_cultivo.eq("transitorio") & camp.campana_completa
                           & (camp.cosecha_ha >= MIN_HA) & (camp.produccion_t > 0))
print(camp.groupby(["campana", "campana_completa"]).registro_valido.agg(["size", "sum"]))

### 6.2 Etiquetas de rendimiento

In [ ]:
val = camp[camp.registro_valido]
ref = (val.groupby("producto").rendimiento_t_ha
       .agg(n="size", mediana="median", p33=lambda s: s.quantile(1/3), p67=lambda s: s.quantile(2/3), p95=lambda s: s.quantile(.95)))
ref = ref[ref.n >= MIN_REG_PRODUCTO]
print("Cultivos con umbrales:", len(ref))

camp = camp.merge(ref[["mediana", "p33", "p67", "p95"]].add_prefix("ref_"), left_on="producto", right_index=True, how="left")
ok = camp.registro_valido & camp.ref_mediana.notna()

camp["rend_relativo"] = np.where(ok, camp.rendimiento_t_ha / camp.ref_mediana, np.nan)
camp["etiqueta_rendimiento"] = np.select(
    [ok & (camp.rendimiento_t_ha <= camp.ref_p33), ok & (camp.rendimiento_t_ha > camp.ref_p67), ok],
    ["bajo", "alto", "medio"], default=None)
camp["exito"] = np.where(ok, (camp.rend_relativo >= 1).astype(float), np.nan)
camp["rend_outlier"] = ok & (camp.rendimiento_t_ha > camp.ref_p95 * 1.5)

print(camp.etiqueta_rendimiento.value_counts(dropna=True), "\n")
print("Exito:", camp.exito.value_counts(normalize=True).round(2).to_dict())
print("Posibles outliers de rendimiento:", camp.rend_outlier.sum())

### 6.3 Etiqueta de perdida (siembra vs cosecha)

In [ ]:
sv = camp.tipo_cultivo.eq("transitorio") & camp.campana_completa & (camp.siembra_ha >= MIN_HA)
camp["ratio_cosecha_siembra"] = np.where(sv, camp.cosecha_ha / camp.siembra_ha.replace(0, np.nan), np.nan)
camp["tasa_perdida_camp"] = (1 - camp.ratio_cosecha_siembra).clip(0, 1)

r = camp.ratio_cosecha_siembra.dropna()
print(r.describe().round(2))
print("Registros con cosecha > siembra:", (r > 1).mean().round(2))
plt.figure(figsize=(8, 3.5))
sns.histplot(r.clip(upper=3), bins=60, color="#b5651d")
plt.axvline(1, color="k", ls="--"); plt.title("Cosecha / siembra por campana (recortado en 3)"); plt.show()

### 6.4 Variables de rotacion a nivel distrito (campana previa)

In [ ]:
n0 = len(camp)
# area transitoria sembrada por distrito y campana; area por familia y por leguminosas
tr = camp[camp.tipo_cultivo.eq("transitorio")]
tot = tr.groupby(["cod_ubigeo", "campana_ini"]).siembra_ha.sum().rename("sup_total").reset_index()
fam = tr.groupby(["cod_ubigeo", "campana_ini", "familia"]).siembra_ha.sum().rename("sup_familia").reset_index()
leg = tr[tr.es_leguminosa].groupby(["cod_ubigeo", "campana_ini"]).siembra_ha.sum().rename("sup_leguminosa").reset_index()

# se desplaza una campana hacia adelante para que represente a la campana previa
for d in (tot, fam, leg):
    d["campana_ini"] += 1

camp = (camp.merge(tot.rename(columns={"sup_total": "prev_sup_total"}), on=["cod_ubigeo", "campana_ini"], how="left")
            .merge(fam.rename(columns={"sup_familia": "prev_sup_familia"}), on=["cod_ubigeo", "campana_ini", "familia"], how="left")
            .merge(leg.rename(columns={"sup_leguminosa": "prev_sup_leguminosa"}), on=["cod_ubigeo", "campana_ini"], how="left"))

# mismo cultivo en la campana previa
prev = camp[["cod_ubigeo", "producto", "campana_ini", "siembra_ha", "rendimiento_t_ha", "etiqueta_rendimiento"]].copy()
prev["campana_ini"] += 1
prev = prev.rename(columns={"siembra_ha": "prev_siembra_cultivo", "rendimiento_t_ha": "prev_rend_cultivo",
                            "etiqueta_rendimiento": "prev_etiqueta_cultivo"})
camp = camp.merge(prev, on=["cod_ubigeo", "producto", "campana_ini"], how="left")

camp["prev_share_familia"] = (camp.prev_sup_familia / camp.prev_sup_total).fillna(0)
camp["prev_share_leguminosa"] = (camp.prev_sup_leguminosa / camp.prev_sup_total).fillna(0)
camp["tiene_campana_previa"] = camp.prev_sup_total.notna()
camp["prev_siembra_cultivo"] = camp.prev_siembra_cultivo.fillna(0)
assert len(camp) == n0, "los merges duplicaron filas"
assert not camp.duplicated(["cod_ubigeo", "producto", "campana"]).any()

print(camp[camp.registro_valido].groupby("campana").tiene_campana_previa.mean().round(2))

## 7. EDA de etiquetas

In [ ]:
m = camp[camp.etiqueta_rendimiento.notna()]
orden = m.groupby("producto").size().sort_values(ascending=False).head(15).index

fig, axs = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(m.rend_relativo.clip(upper=3), bins=50, ax=axs[0], color="#4c9a2a")
axs[0].axvline(1, color="k", ls="--"); axs[0].set_title("Rendimiento relativo a la mediana del cultivo")

ct = (m[m.producto.isin(orden)].groupby(["producto", "etiqueta_rendimiento"]).size().unstack(fill_value=0)
      .reindex(orden)[["bajo", "medio", "alto"]])
ct.plot.barh(stacked=True, ax=axs[1], color=["#d9534f", "#f0ad4e", "#5cb85c"])
axs[1].set_title("Clases de rendimiento por cultivo"); axs[1].set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
# rendimiento relativo por provincia y por campana
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
sns.boxplot(data=m, x="provincia", y="rend_relativo", ax=axs[0], color="#8fc17a", fliersize=2)
axs[0].set_ylim(0, 3); axs[0].tick_params(axis="x", rotation=40); axs[0].set_title("Rendimiento relativo por provincia")
sns.boxplot(data=m, x="campana", y="rend_relativo", ax=axs[1], color="#8fc17a", fliersize=2)
axs[1].set_ylim(0, 3); axs[1].set_title("Rendimiento relativo por campana")
plt.tight_layout(); plt.show()

In [ ]:
# ingreso por hectarea para cultivos con mayor numero de registros validos
ing = m[m.producto.isin(orden) & m.ingreso_ha_soles.notna()]
plt.figure(figsize=(10, 5))
sns.boxplot(data=ing, y="producto", x="ingreso_ha_soles", order=orden, color="#e0b36a", fliersize=2)
plt.xscale("log"); plt.title("Ingreso bruto por hectarea (S/, escala log)"); plt.ylabel(""); plt.show()

## 8. Tabla final de modelado

In [ ]:
cols_modelo = [
    # identificadores
    "cod_ubigeo", "provincia", "distrito", "producto", "campana", "campana_ini",
    # cultivo
    "grupo", "familia", "es_leguminosa", "mes_pico_siembra",
    # campana actual
    "siembra_ha", "cosecha_ha", "produccion_t", "precio_kg", "n_meses_activo",
    # rotacion (campana previa)
    "tiene_campana_previa", "prev_siembra_cultivo", "prev_share_familia", "prev_share_leguminosa",
    "prev_rend_cultivo", "prev_etiqueta_cultivo",
    # etiquetas
    "rendimiento_t_ha", "rend_relativo", "etiqueta_rendimiento", "exito", "ingreso_ha_soles",
    "ratio_cosecha_siembra", "tasa_perdida_camp", "rend_outlier",
]
modelo = camp[camp.etiqueta_rendimiento.notna()][cols_modelo].reset_index(drop=True)
print(modelo.shape)
print(modelo.isna().mean().round(2)[lambda s: s > 0])
modelo.head()

## 9. Exportacion

In [ ]:
mensual.to_csv("sisagri_mensual_limpio.csv", index=False, encoding="utf-8-sig")
camp.to_csv("sisagri_campana.csv", index=False, encoding="utf-8-sig")
modelo.to_csv("sisagri_modelo.csv", index=False, encoding="utf-8-sig")
print("mensual:", mensual.shape, "| campana:", camp.shape, "| modelo:", modelo.shape)

try:
    from google.colab import files
    for f in ["sisagri_mensual_limpio.csv", "sisagri_campana.csv", "sisagri_modelo.csv"]:
        files.download(f)
except ImportError:
    pass